## 1) Setup

In [51]:

# If needed, install once:
# %pip install --upgrade sentence-transformers numpy pandas tqdm
# For local LLM via Ollama:
# %pip install --upgrade requests
# Vector store
# %pip install --upgrade chromadb

### PreDev Setup

In [52]:
from importlib import reload  # Reload modules during development
import os  # OS utilities
import requests  # HTTP requests
import numpy as np  # Numerical operations
import faiss  # Vector similarity search

import database  # Local database module
from database import AmberChromaAPI  # Amber-Chroma interface

from pypdf import PdfReader  # PDF reading
from sentence_transformers import SentenceTransformer  # Text embeddings

reload(database)  # Refresh module changes


<module 'database' from '/home/bsauce11/RAG_Prototype/Code_Saucedo/My_PreDev/database.py'>

In [53]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

### Document Splitting

In [54]:
# # Initialize text splitter
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,  # Maximum size of each chunk
#     chunk_overlap=50,  # Overlap between chunks to maintain context
#     length_function=len,
#     separators=[" "]  # Hierarchy of separators
# )
# chunks=text_splitter.split_documents(documents)
#
# print(f"Created {len(chunks)} chunks from {len(documents)} documents")
# print(f"\nChunk example:")
# print(f"Content: {chunks[0].page_content[:150]}...")
# print(f"Metadata: {chunks[0].metadata}")

In [55]:
# chunks

### Embedding Models

In [56]:
### Huggingface model

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [57]:
# vector=embeddings.embed_query(sample_text)
# vector

### Amber ChromaDB

In [58]:
# Chroma DB instance
API_CHROMA_DB = AmberChromaAPI(db_path="/opt/chromadb/data/prompt_db")
# Embedding model
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
# PDF file path
PDF_ADDRESS = "Amber25.pdf"
# Local Ollama server URL
OLLAMA_URL = "http://127.0.0.1:11434"


Using local ChromaDB path: /opt/chromadb/data/prompt_db


In [59]:
threshold_ChromaDB = 0.35 # Similarity threshold for Mails
threshold_PDF = 0.45  # Similarity threshold for PDF results

### Chunking

In [60]:
# # Retrieve relevant chunks from hybrid retriever
# chunks = retrieve_with_pdf(
#     question,
#     k_chroma=50,     # Number of ChromaDB results
#     k_pdf=5,         # Number of PDF results
#     threshold=threshold_ChromaDB    # Similarity threshold
# )

In [61]:
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# If you already have this client, reuse it:
client = chromadb.PersistentClient(path="/opt/chromadb/data/prompt_db")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    client=client,
    collection_name="rag_collection",
    embedding_function=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [62]:
# ## Create a Chromdb vector store
# persist_directory="/opt/chromadb/data/prompt_db"
#
# ## Initialize Chromadb with HuggingFace embeddings
# vectorstore=Chroma.from_documents(
#     documents=chunks,
#     embedding=HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"),
#     persist_directory=persist_directory,
#     collection_name="rag_collection"
#
# )
#
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Vector store name: {vectorstore._collection.name}")

Vector store created with 0 vectors
Vector store name: rag_collection


### Test Similarity Search

In [63]:
query="What is Amber?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[]

In [64]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What is Amber?

Top 0 similar chunks:


### Advanced Similarity Search With Scores

In [65]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [66]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)
llm

ChatOllama(model='llama3.1:8b', temperature=0.0)

In [67]:
# from langchain_community.llms import Ollama
#
# #OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
# #OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
#
# llm = Ollama(model="llama3.1:8b")

In [68]:
# Use the LLM to generate prompt
test_response=llm.invoke("What is Amber?")
test_response

AIMessage(content='Amber is a fascinating substance with a rich history. Here\'s what it is:\n\n**Definition:** Amber is a fossilized tree resin that has been hardened over time, often containing ancient plant and animal remains.\n\n**Formation:** Amber forms when pine trees or other conifers produce sticky resin to protect themselves from insects, diseases, and environmental stressors. This resin can seep out of the tree\'s bark and harden in place, trapping small organisms like insects, spiders, and even tiny mammals within its sticky matrix.\n\n**Properties:** Amber is a translucent, yellowish-brown or golden-colored substance with a waxy texture. It has a distinctive "glow" due to the way it refracts light. Amber can be found in various forms, including:\n\n1. **Fossilized resin**: The original tree resin that has hardened over time.\n2. **Amberite**: A type of amber that contains more than 10% of organic matter, such as plant or animal remains.\n3. **Baltic amber**: A specific typ

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [69]:
# ## Convert vector store to retriever
# retriever=vectorstore.as_retriever(
#      search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
#  )
# retriever

In [70]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [71]:
SYSTEM_PROMPT = """
You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
7) Do NOT include citation markers, chunk labels, similarity scores,
   reference numbers, or metadata in your output.
8) Do NOT repeat metadata from the Context.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer and
Include citations, references, or metadata.
"""

In [72]:
"""
You are a concise, technical assistant for Amber molecular simulation users. "
    Answer the user's question using ONLY the provided context.
    If the answer cannot be determined from the context, say you do not know.
    Cite sources using [Title#chunkN] notation.
    Do not speculate or introduce external knowledge.

    Do not include citations, references, or metadata.

    Output the final answer.
For source citations, include the source page number (for PDF) or source label (for archive) at the beginning.

"""

'\nYou are a concise, technical assistant for Amber molecular simulation users. "\n    Answer the user\'s question using ONLY the provided context.\n    If the answer cannot be determined from the context, say you do not know.\n    Cite sources using [Title#chunkN] notation.\n    Do not speculate or introduce external knowledge.\n\n    Do not include citations, references, or metadata.\n'

In [73]:
from langchain_core.prompts import ChatPromptTemplate

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer and Include citations, references, or metadata.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4) If the Context contains relevant information, use it to answer as completely as possible.\n5) Do NOT mention an Persona, Identity, or Role in your answer.\n6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.\n\nSTYLE-\n- Start with a clear, d

In [74]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwargs={"k":3} ## Retrieve top 3 relevant chunks
 )

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1503be4a2d20>, search_kwargs={'k': 3})

In [75]:
# ## Format the output documents for the prompt
# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)

In [76]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", "unknown_page")
        chunk = doc.metadata.get("chunk_id", i)
        formatted.append(
            f"[Source: {source} | Page: {page} | Chunk: {chunk}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [77]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1503be4a2d20>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4

In [78]:
response=rag_chain_lcel.invoke("What is Amber")
response

"The AMBER (Assisted Model Building with Energy Refinement) molecular dynamics suite is a software package used for simulating the behavior of molecules. It was originally developed by Peter Kollman's group at the University of California, San Francisco (UCSF).\n\n[Citation: Pearlman et al., 1995]\n\nTechnical Explanation:\nThe AMBER suite includes tools for preparing molecular structures, running molecular dynamics simulations, and analyzing simulation results. It uses a combination of empirical energy functions and quantum mechanics to model the behavior of molecules.\n\n[Reference: Case et al., 2012]\n\nPractical Guidance:\nTo use the AMBER suite, users typically need to prepare their system by creating an input file that specifies the molecular structure, force field parameters, and simulation conditions. The `tleap` program is often used for this purpose.\n\n[Citation: Leach, 2001]"

In [79]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is Amber")


In [80]:
#retriever.get_relevant_documents("What is Deep Learning")
retDocs = retriever.invoke("What is Deep Learning")
retDocs

[]

In [81]:
# Query using the LCEL approach
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    #docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
        print(doc.metadata)

In [82]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What can I use Amber for?")

Testing LCEL Chain:
Question: What can I use Amber for?
--------------------------------------------------
Answer: **Molecular Dynamics Simulations**

You can use Amber to perform molecular dynamics (MD) simulations of biomolecules, such as proteins, nucleic acids, and lipids. This includes:

* Equilibration and production runs
* Free energy calculations
* Molecular mechanics and molecular dynamics simulations
* Simulation analysis and visualization

**Technical Explanation**

Amber is a widely used software package for molecular dynamics simulations that utilizes the generalized Born implicit solvent model (GB/SA) and the AMOEBA polarizable force field. It can be used to study various aspects of biomolecular behavior, including protein-ligand interactions, protein folding, and membrane simulations.

**Practical Guidance**

For a basic MD simulation using Amber, you will need to create an input file (.prmtop and .inpcrd) and run the simulation using the pmemd or sander executable. Cons

In [83]:
query_rag_lcel("Why should SHAKE be disabled during minimization in AMBER?")

Question: Why should SHAKE be disabled during minimization in AMBER?
--------------------------------------------------
Answer: **Final Answer:** SHAKE should be disabled during minimization because it can lead to inaccurate energy calculations and convergence issues.

**Technical Explanation:** SHAKE is a constraint that fixes the bond lengths of hydrogen atoms, which can cause problems during minimization. When SHAKE is enabled, it can lead to incorrect energy calculations due to the artificial constraints on the system. Additionally, SHAKE can hinder the convergence of the minimization process by preventing the system from exploring its full conformational space.

**Practical Guidance:** To disable SHAKE during minimization in AMBER, use the following command: `:SHAKE=off` (1). This will allow the system to relax and minimize without the constraints imposed by SHAKE.

Reference:
(1) Case et al. (2005). The Amber Biomolecular Simulation Programs. Journal of Computational Chemistry, 2

In [84]:
query_rag_lcel("How does AMBER handle time-series anomaly detection?")

Question: How does AMBER handle time-series anomaly detection?
--------------------------------------------------
Answer: AMBER uses the "anomalous" flag in the mdin file to identify anomalous frames. The "anomalous" flag is set to 1 for frames that are considered anomalous, and 0 otherwise.

Technical Explanation:
The "anomalous" flag is used in conjunction with the "anomalous" keyword in the mdin file (1). When this keyword is present, AMBER will identify frames as anomalous based on a user-specified threshold. The threshold value can be set using the "anomalous-threshold" keyword.

Practical Guidance:
To use time-series anomaly detection in AMBER, create an mdin file with the "anomalous" and "anomalous-threshold" keywords specified. For example:

```
&mdin
  anomalous = 1
  anomalous-threshold = 10.0
/
```

This will identify frames as anomalous if their energy exceeds 10.0 kcal/mol above the average energy.

Reference:
(1) Case, D. A., et al. (2005). AMBER 8: United States. Univers

In [85]:
query_rag_lcel("How can I get SHAKE to consider two different residue names to be water?")

Question: How can I get SHAKE to consider two different residue names to be water?
--------------------------------------------------
Answer: To make SHAKE consider two different residue names as water, you need to specify them in the `BONDS` section of your input file. 

You can use the `WAT` keyword followed by the residue name(s) you want to treat as water. For example:

```
&cntrl
  ...
  BONDS = WAT HETATM
/
```

This will tell SHAKE to apply the water constraints to both "HETATM" and any other residues specified in the `WAT` keyword.

Technical Explanation:
The `BONDS` section is used to specify which bonds should be treated as flexible or rigid. The `WAT` keyword is a special case that tells SHAKE to treat the specified residue(s) as water, applying the corresponding constraints. This allows you to use different residue names for water molecules while still taking advantage of SHAKE's efficiency.

Reference:
[1] Pearlman et al., "AMBER, a package for molecular dynamics simulatio

In [86]:
query_rag_lcel("How can I add the CG protein into a CG bilayer?")

Question: How can I add the CG protein into a CG bilayer?
--------------------------------------------------
Answer: To add the CG protein into a CG bilayer, you can use the `edit` command in AmberTools to insert the protein molecule into the existing bilayer system. 

First, make sure that the protein and bilayer systems are already prepared and saved as separate files. Then, open the bilayer file in the editor and use the `insert` option to add the protein molecule at a specified position.

For example:
```
edit -i protein.pdb -o system.pdb insert
```
This will insert the protein molecule from the `protein.pdb` file into the current system file, which is being edited. The resulting system will be saved as `system.pdb`.

Technical Explanation:

The `edit` command in AmberTools allows for interactive editing of molecular systems. The `-i` option specifies the input file containing the protein molecule to be inserted, and the `-o` option specifies the output file where the modified syst

In [87]:
query_rag_lcel("How do I obtain a Z-DNA structure from NAB?")

Question: How do I obtain a Z-DNA structure from NAB?
--------------------------------------------------
Answer: To obtain a Z-DNA structure from NAB (Nucleic Acid Builder), you can use the following steps:

1. Run `nab` to build your DNA sequence.
2. Use the `z-dna` option in the `nab` input file to specify that you want to generate a Z-DNA structure.

Technical Explanation:
The `z-dna` option is used to generate a Z-DNA structure from the nucleic acid sequence built by NAB. This option can be specified in the input file for NAB, and it will modify the DNA structure to adopt a Z-DNA conformation.

Practical Guidance:
Make sure to include the `z-dna` option in your NAB input file, and also specify any other relevant parameters such as the sequence length, temperature, and salt concentration. You can then use the generated Z-DNA structure as an input for further simulations or analysis using AMBER tools.

Reference: 
The `nab` manual (available at <https://ambermd.org/ManualCurrent.pdf>

In [88]:
query_rag_lcel("If we do not neutralize our system of protein-ligand complex properly or the addition of ions is not appropriate, will the properties like planarity of the system be affected?")

Question: If we do not neutralize our system of protein-ligand complex properly or the addition of ions is not appropriate, will the properties like planarity of the system be affected?
--------------------------------------------------
Answer: **Yes**, if the system is not properly neutralized or ions are not added appropriately, it can affect the properties of the system, including planarity.

The AMBER manual states that "neutralization of the system is crucial to ensure accurate calculations" [1]. If the system is not neutralized correctly, it can lead to artifacts in the simulation results. Similarly, if ions are not added properly, it can also affect the properties of the system.

In particular, planarity of the system may be affected due to electrostatic interactions between charged groups and the ligand. The AMBER manual notes that "electrostatic interactions can significantly impact the conformational space sampled by the system" [2]. If these interactions are not properly acc

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [89]:
# from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
# from langchain_core.prompts import MessagesPlaceholder
# from langchain_core.messages import HumanMessage, AIMessage

In [90]:
# ## create a prompt that includes the chat history
# contextualize_q_system_prompt = """Given a chat history and the latest user question
# which might reference context in the chat history, formulate a standalone question
# which can be understood without the chat history. Do NOT answer the question,
# just reformulate it if needed and otherwise return it as is."""
#
# contextualize_q_prompt = ChatPromptTemplate.from_messages([
#     ("system", contextualize_q_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])

In [91]:
# ## create history aware retriever
# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
# history_aware_retriever

In [92]:
# from langchain_classic.chains.retrieval import create_retrieval_chain
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
#
# # Create a new document chain with history
# qa_system_prompt = """You are an assistant for question-answering tasks.
# Use the following pieces of retrieved context to answer the question.
# If you don't know the answer, just say that you don't know.
# Use three sentences maximum and keep the answer concise.
#
# Context: {context}"""
#
# qa_prompt = ChatPromptTemplate.from_messages([
#     ("system", qa_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])
#
# question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
#
# # Create conversational RAG chain
# conversational_rag_chain = create_retrieval_chain(
#     history_aware_retriever,
#     question_answer_chain
# )
# print("Conversational RAG chain created!")

In [93]:
# chat_history=[]
# # First question
# result1 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What is machine learning?"
# })
# print(f"Q: What is machine learning?")
# print(f"A: {result1['answer']}")

In [94]:
# chat_history.extend([
#     HumanMessage(content="What is machine learning"),
#     AIMessage(content=result1['answer'])
# ])

In [95]:
# chat_history

In [96]:
# ## Follow up question
# # Follow-up question
# result2 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What are its main types?"  # Refers to ML from previous question
# })
# result2

In [97]:
# result2['answer']